## Gold Layer: 7日移动平均 GMV

**职责**: 从 Silver 层读取订单宽表，计算每日 GMV 及其 7 日滑动均值。

**为什么用 PySpark DataFrame API 而不是 SQL？**
- 这是 Spark 学习实验的核心价值：展示 PySpark DataFrame API 的窗口函数能力
- dbt 中已有 `mart_daily_gmv_trend`（SQL 实现），本 notebook 用 PySpark API 实现相同的分析逻辑
- 面试叙事：**同一个指标，两种技术栈实现——SQL (dbt) vs DataFrame API (Spark)**

**输入**: `silver_fact_orders`
**输出**: `gold_gmv_7day_moving_avg` Delta 表

In [0]:
# ============================================================
# 导入 PySpark 函数库
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# ============================================================
# 从 Silver 层读取数据
# ============================================================

# 读取 silver_fact_orders，只取计算 GMV 所需的列
df = spark.table("silver_fact_orders").select(
    "order_id",
    "order_purchase_timestamp",
    "price"
).filter(
    F.col("order_purchase_timestamp").isNotNull()
)

print(f"📥 读取 Silver 数据，共 {df.count():,} 行")

In [0]:
# ============================================================
# 计算每日 GMV（按日期聚合）
# 使用 PySpark DataFrame API 的 groupBy 和 agg
# ============================================================

# 将时间戳转为日期（只保留年月日），方便按天分组
daily_gmv = (
    df
    .withColumn("order_date", F.to_date("order_purchase_timestamp"))
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("daily_orders"),
        F.sum("price").alias("daily_gmv")
    )
    .orderBy("order_date")
)

print("📊 每日 GMV 聚合完成，前 5 行预览：")
daily_gmv.show(5, truncate=False)

In [0]:
# ============================================================
# 🎯 核心：PySpark DataFrame API 窗口函数 — 7 日移动平均
# ============================================================

# 定义窗口：以 order_date 排序，取当前行及其前 6 行（共 7 行）
# ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
window_7days = Window.orderBy("order_date").rowsBetween(-6, 0)

# 在窗口上计算 7 日移动平均
gold_gmv_7day = daily_gmv.withColumn(
    "gmv_7day_moving_avg",
    F.round(F.avg("daily_gmv").over(window_7days), 2)
).withColumn(
    "orders_7day_moving_avg",
    F.round(F.avg("daily_orders").over(window_7days), 2)
)

print("📈 7 日移动平均计算完成，预览：")
gold_gmv_7day.show(10, truncate=False)

In [0]:
# ============================================================
# 写入 Gold 层 Delta 表
# ============================================================

gold_gmv_7day.write.format("delta").mode("overwrite").saveAsTable("gold_gmv_7day_moving_avg")

row_count = gold_gmv_7day.count()
print(f"✅ gold_gmv_7day_moving_avg 写入完成 — {row_count:,} 天数据")